# 📊 01 — Exploratory Data Analysis (EDA)
**Startup Funding Analysis Project**  
This notebook performs a comprehensive Exploratory Data Analysis on the raw Startup Funding dataset.  
We will understand the shape, quality, distributions, trends and patterns before any modelling.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Premium dark theme
plt.rcParams['figure.facecolor'] = '#0f0f1a'
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['axes.edgecolor'] = '#3a3a5c'
plt.rcParams['axes.labelcolor'] = '#c0c0e0'
plt.rcParams['xtick.color'] = '#a0a0c0'
plt.rcParams['ytick.color'] = '#a0a0c0'
plt.rcParams['text.color'] = '#e0e0f0'
plt.rcParams['grid.color'] = '#2a2a4a'
plt.rcParams['grid.alpha'] = 0.5
plt.rcParams['font.family'] = 'DejaVu Sans'

PALETTE = ['#5B8DEF', '#8E5BEF', '#EF5B8D', '#EFB85B', '#5BEFB8', '#EF8E5B']

print('✅ Libraries loaded successfully!')

## 2. Load Raw Dataset

In [ ]:
# Load the raw dataset
DATA_PATH = '../data/raw/Startup_Funding_Cleaned.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f'✅ Dataset loaded successfully!')
print(f'   Rows    : {df.shape[0]:,}')
print(f'   Columns : {df.shape[1]}')
df.head(5)

## 3. Dataset Overview

In [ ]:
# Column data types and non-null counts
print('=== Column Data Types & Non-Null Counts ===')
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
info_df = info_df.sort_values('null_pct', ascending=False)
print(info_df.to_string())

In [ ]:
# Statistical summary for numeric columns
print('=== Statistical Summary (Numeric Columns) ===')
df.describe().T.style.background_gradient(cmap='Blues')

## 4. Missing Values Analysis

In [ ]:
# Visualize missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(missing.index, missing_pct.values, color=PALETTE[0], alpha=0.85, edgecolor='#3a3a5c')

# Add value labels
for bar, pct in zip(bars, missing_pct.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', ha='left', color='#c0c0e0', fontsize=9)

ax.set_xlabel('Missing Percentage (%)', labelpad=10)
ax.set_title('Missing Values by Column', fontsize=16, fontweight='bold', pad=15)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/missing_values.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Total columns with missing values: {len(missing)}')

## 5. Duplicate Records Check

In [ ]:
total_rows = len(df)
duplicate_rows = df.duplicated().sum()
print(f'Total rows              : {total_rows:,}')
print(f'Duplicate rows          : {duplicate_rows:,}')
print(f'Duplicate percentage    : {duplicate_rows/total_rows*100:.2f}%')

# Check unique startups and funding rounds
if 'company_id' in df.columns:
    print(f'Unique Companies        : {df["company_id"].nunique():,}')
if 'funding_round_id' in df.columns:
    print(f'Unique Funding Rounds   : {df["funding_round_id"].nunique():,}')

## 6. Funding Amount Distribution

In [ ]:
# Analyze funding amount distribution
funding_col = 'raised_amount_usd'
if funding_col in df.columns:
    funding_data = df[funding_col].dropna()
    funding_data = funding_data[funding_data > 0]
    log_funding = np.log1p(funding_data)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Raw distribution
    axes[0].hist(funding_data.clip(upper=funding_data.quantile(0.99)),
                 bins=60, color=PALETTE[0], alpha=0.85, edgecolor='#3a3a5c')
    axes[0].set_title('Funding Amount Distribution (Raw, clipped at P99)', fontweight='bold')
    axes[0].set_xlabel('Raised Amount (USD)')
    axes[0].set_ylabel('Frequency')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))

    # Log-normal distribution
    axes[1].hist(log_funding, bins=60, color=PALETTE[1], alpha=0.85, edgecolor='#3a3a5c')
    axes[1].set_title('Log-Transformed Funding Amount (Near Normal)', fontweight='bold')
    axes[1].set_xlabel('Log(1 + Raised Amount USD)')
    axes[1].set_ylabel('Frequency')

    for ax in axes:
        ax.grid(alpha=0.3)

    plt.suptitle('Funding Amount Distribution Analysis', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    os.makedirs('../visualizations', exist_ok=True)
    plt.savefig('../visualizations/funding_distribution.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

    print(f'Min Funding     : ${funding_data.min():,.0f}')
    print(f'Median Funding  : ${funding_data.median():,.0f}')
    print(f'Mean Funding    : ${funding_data.mean():,.0f}')
    print(f'Max Funding     : ${funding_data.max():,.0f}')

## 7. Funding Round Types Analysis

In [ ]:
round_col = 'funding_round_type'
if round_col in df.columns:
    round_counts = df[round_col].value_counts().head(12)

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(round_counts)))
    bars = ax.bar(round_counts.index, round_counts.values, color=colors, edgecolor='#3a3a5c', alpha=0.9)

    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
                f'{bar.get_height():,}', ha='center', va='bottom', fontsize=9, color='#c0c0e0')

    ax.set_title('Distribution of Funding Round Types', fontsize=15, fontweight='bold', pad=15)
    ax.set_xlabel('Funding Round Type', labelpad=10)
    ax.set_ylabel('Number of Records', labelpad=10)
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../visualizations/funding_round_types.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 8. Temporal Funding Trends

In [ ]:
if 'funded_at' in df.columns:
    df['funded_at'] = pd.to_datetime(df['funded_at'], errors='coerce')
    df['funding_year'] = df['funded_at'].dt.year

    yearly = df.groupby('funding_year').agg(
        num_deals=('funding_year', 'count'),
        total_funding=('raised_amount_usd', 'sum')
    ).dropna().reset_index()
    yearly = yearly[(yearly['funding_year'] >= 1995) & (yearly['funding_year'] <= 2025)]

    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    # Deal count
    axes[0].fill_between(yearly['funding_year'], yearly['num_deals'], alpha=0.3, color=PALETTE[0])
    axes[0].plot(yearly['funding_year'], yearly['num_deals'], color=PALETTE[0], linewidth=2.5, marker='o', markersize=4)
    axes[0].set_title('Annual Number of Funding Deals', fontweight='bold', fontsize=13)
    axes[0].set_ylabel('Number of Deals')
    axes[0].grid(alpha=0.3)
    # Recession highlight
    axes[0].axvspan(2008, 2009, alpha=0.15, color='red', label='Recession Era 2008-09')
    axes[0].legend()

    # Total funding
    axes[1].fill_between(yearly['funding_year'], yearly['total_funding'] / 1e9, alpha=0.3, color=PALETTE[1])
    axes[1].plot(yearly['funding_year'], yearly['total_funding'] / 1e9, color=PALETTE[1], linewidth=2.5, marker='s', markersize=4)
    axes[1].set_title('Annual Total Funding (USD Billions)', fontweight='bold', fontsize=13)
    axes[1].set_ylabel('Total Funding ($B)')
    axes[1].set_xlabel('Year')
    axes[1].grid(alpha=0.3)
    axes[1].axvspan(2008, 2009, alpha=0.15, color='red')

    plt.suptitle('Startup Funding Temporal Trends (1995–2025)', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../visualizations/temporal_trends.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 9. Industry Sector Analysis

In [ ]:
sector_col = 'Industry_Sector'
if sector_col in df.columns:
    sector_counts = df[sector_col].value_counts().head(15)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Bar chart
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(sector_counts)))
    axes[0].barh(sector_counts.index[::-1], sector_counts.values[::-1], color=colors[::-1], alpha=0.9, edgecolor='#3a3a5c')
    axes[0].set_title('Top 15 Industry Sectors by Deal Count', fontweight='bold', fontsize=13)
    axes[0].set_xlabel('Number of Deals')
    axes[0].grid(axis='x', alpha=0.3)

    # Pie chart (top 8)
    top8 = sector_counts.head(8)
    explode = [0.05] * len(top8)
    wedges, texts, autotexts = axes[1].pie(
        top8.values, labels=top8.index, autopct='%1.1f%%',
        startangle=140, pctdistance=0.8,
        explode=explode,
        colors=plt.cm.plasma(np.linspace(0.2, 0.9, len(top8)))
    )
    for text in autotexts:
        text.set_fontsize(8)
        text.set_color('#ffffff')
    axes[1].set_title('Industry Sector Market Share (Top 8)', fontweight='bold', fontsize=13)

    plt.suptitle('Industry Sector Distribution Analysis', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visualizations/sector_analysis.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 10. Geographic Distribution

In [ ]:
if 'country_code' in df.columns:
    country_counts = df['country_code'].value_counts().head(15)

    fig, ax = plt.subplots(figsize=(14, 6))
    colors = plt.cm.cool(np.linspace(0.2, 0.9, len(country_counts)))
    bars = ax.bar(country_counts.index, country_counts.values, color=colors, alpha=0.9, edgecolor='#3a3a5c')

    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                f'{bar.get_height():,}', ha='center', va='bottom', fontsize=8, color='#c0c0e0')

    ax.set_title('Top 15 Countries by Number of Startup Funding Deals', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Country Code', labelpad=10)
    ax.set_ylabel('Number of Deals', labelpad=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../visualizations/geographic_distribution.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 11. Startup Status Distribution

In [ ]:
status_col = 'Startup_Status'
if status_col in df.columns:
    status_counts = df.drop_duplicates('company_id')[status_col].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(status_counts))]
    axes[0].bar(status_counts.index, status_counts.values, color=colors, edgecolor='#3a3a5c', alpha=0.9)
    for i, (label, val) in enumerate(status_counts.items()):
        axes[0].text(i, val + 10, f'{val:,}', ha='center', fontsize=10, color='#c0c0e0')
    axes[0].set_title('Startup Status Distribution (Unique Companies)', fontweight='bold')
    axes[0].set_xlabel('Status')
    axes[0].set_ylabel('Count')
    axes[0].grid(axis='y', alpha=0.3)

    # Donut chart
    wedges, texts, autotexts = axes[1].pie(
        status_counts.values, labels=status_counts.index,
        autopct='%1.1f%%', startangle=90,
        colors=colors,
        wedgeprops=dict(width=0.55)
    )
    axes[1].set_title('Startup Outcome Breakdown (Donut)', fontweight='bold')

    plt.suptitle('Startup Status & Outcome Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visualizations/startup_status.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

    # Success rate
    success_count = status_counts.get('acquired', 0) + status_counts.get('ipo', 0)
    total_unique = status_counts.sum()
    print(f'\n📊 Success Rate (Acquired + IPO): {success_count}/{total_unique} = {success_count/total_unique*100:.2f}%')

## 12. Top Investors Analysis

In [ ]:
investor_col = 'Investor_Name'
if investor_col in df.columns:
    investors = df[investor_col].dropna()
    investors = investors[~investors.str.lower().str.contains('undisclosed', na=False)]
    top_investors = investors.value_counts().head(15)

    fig, ax = plt.subplots(figsize=(14, 7))
    colors = plt.cm.plasma(np.linspace(0.25, 0.9, len(top_investors)))
    bars = ax.barh(top_investors.index[::-1], top_investors.values[::-1], color=colors, alpha=0.9, edgecolor='#3a3a5c')

    for bar in bars:
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                f'{bar.get_width():,.0f}', va='center', ha='left', fontsize=9, color='#c0c0e0')

    ax.set_title('Top 15 Most Active Investors (by Number of Deals)', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Number of Investment Deals')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../visualizations/top_investors.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 13. Average Funding by Sector

In [ ]:
if 'Industry_Sector' in df.columns and 'raised_amount_usd' in df.columns:
    sector_funding = df.groupby('Industry_Sector')['raised_amount_usd'].agg(['mean', 'median', 'sum']).reset_index()
    sector_funding.columns = ['Sector', 'Mean_Funding', 'Median_Funding', 'Total_Funding']
    sector_funding = sector_funding[sector_funding['Total_Funding'] > 0].sort_values('Median_Funding', ascending=False).head(12)

    fig, ax = plt.subplots(figsize=(14, 7))
    x = np.arange(len(sector_funding))
    width = 0.35

    bars1 = ax.bar(x - width/2, sector_funding['Mean_Funding'] / 1e6, width, label='Mean Funding', color=PALETTE[0], alpha=0.85, edgecolor='#3a3a5c')
    bars2 = ax.bar(x + width/2, sector_funding['Median_Funding'] / 1e6, width, label='Median Funding', color=PALETTE[2], alpha=0.85, edgecolor='#3a3a5c')

    ax.set_xticks(x)
    ax.set_xticklabels(sector_funding['Sector'], rotation=40, ha='right', fontsize=9)
    ax.set_title('Average vs Median Funding by Industry Sector (Top 12)', fontsize=14, fontweight='bold', pad=15)
    ax.set_ylabel('Funding Amount (USD Millions)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../visualizations/funding_by_sector.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 14. Funding Distribution by Top Countries (Box Plot)

In [ ]:
if 'country_code' in df.columns and 'raised_amount_usd' in df.columns:
    top_countries_list = df['country_code'].value_counts().head(8).index.tolist()
    box_data = df[df['country_code'].isin(top_countries_list) & (df['raised_amount_usd'] > 0)].copy()
    box_data['log_funding'] = np.log1p(box_data['raised_amount_usd'])

    fig, ax = plt.subplots(figsize=(14, 6))
    country_groups = [box_data[box_data['country_code'] == c]['log_funding'].dropna().values for c in top_countries_list]
    
    bp = ax.boxplot(country_groups, labels=top_countries_list, patch_artist=True, notch=True)
    colors_box = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_countries_list)))
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title('Log Funding Distribution by Top 8 Countries', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Country Code')
    ax.set_ylabel('Log(1 + Funding USD)')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../visualizations/funding_by_country_box.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 15. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

if not numeric_df.empty:
    corr_matrix = numeric_df.corr()

    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        corr_matrix, mask=mask, annot=True, fmt='.2f',
        cmap='RdYlBu', center=0, vmin=-1, vmax=1,
        linewidths=0.5, linecolor='#1a1a2e',
        cbar_kws={'shrink': 0.8},
        ax=ax
    )
    ax.set_title('Correlation Heatmap — Numeric Features', fontsize=15, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('../visualizations/correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 16. EDA Summary

In [ ]:
print('='*60)
print('         EDA COMPLETE — KEY FINDINGS SUMMARY')
print('='*60)
print(f'\n📁 Dataset Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'🏢 Unique Startups       : {df["company_id"].nunique() if "company_id" in df.columns else "N/A":,}')

if 'raised_amount_usd' in df.columns:
    total_usd = df['raised_amount_usd'].sum()
    print(f'💰 Total Capital Deployed: ${total_usd/1e9:.2f} Billion')

if 'funding_year' in df.columns:
    print(f'📅 Year Range            : {int(df["funding_year"].min())} – {int(df["funding_year"].max())}')

print('\n📌 Key Observations:')
print('   • Funding amounts are heavily right-skewed → log transformation needed for ML')
print('   • USA dominates globally, followed by GBR, IND, and CAN')
print('   • Software, Biotech, and Mobile are top funded sectors')
print('   • Recession era 2008-09 shows clear dip in deal volumes')
print('   • Significant class imbalance: most startups remain private/operating')
print('\n✅ Ready for Data Cleaning (Notebook 02)')